## Queries - ComCam

In this notebook, we show how to query the ComCam repository\
and view the resulting images.\
Craig Lage - 17-Nov-24

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.io.fits as pf
from lsst.daf.butler import Butler
import lsst.summit.utils.butlerUtils as butlerUtils
from lsst.ip.isr import IsrTask, IsrTaskConfig
from lsst.summit.utils.plotting import plot
import lsst.afw.cameraGeom.utils as camGeomUtils
from lsst.geom import Point2D, Extent2I

In [ ]:
butler = butlerUtils.makeDefaultButler("LSSTComCam")
instrument = 'LSSTComCam'

## First, get a list of exposures
### These should match what you see in RubinTV.

In [ ]:
dayObs = 20241116
instrument = "LSSTComCam"

exposureList = []
for record in butler.registry.queryDimensionRecords("exposure", 
                    where=f"exposure.day_obs={dayObs} and instrument='LSSTComCam'"):
    exposureList.append([record.id, record])
exposureList.sort(key=lambda x: x[0])
for [id,record] in exposureList:
    print(record.id, record.observation_type, record.exposure_time, record.physical_filter)


# Get the data from the headers

In [ ]:
expId = 2024111600300
mData = butler.get('raw.metadata', detector=4, exposure=expId, instrument=instrument)
for key in mData.keys():
    print(key, mData[key])

# Now get the image data

## Next, look at the raw data from one of the exposures.
### There are generally three options, raw, postISRCCD, and calexp
### Each has successively more processing

In [ ]:
expId = 2024111600300
#exp = butler.get('raw', detector=4, exposure=expId, instrument=instrument)
#exp = butler.get('postISRCCD', detector=4, exposure=expId, instrument=instrument)
exp = butler.get('calexp', detector=4, visit=expId, instrument=instrument)
%matplotlib inline        
x = plot(exp, stretch='ccs')

## Define a simple ISR
## Maybe you want to start with the raw image and do your own ISR
### This has just overscan subtraction and bias subtraction.

In [ ]:
isrConfig = IsrTaskConfig()
isrConfig.doLinearize=False
isrConfig.doOverscan=True
isrConfig.overscan.fitType="MEDIAN_PER_ROW"
isrConfig.overscan.doParallelOverscan=True
isrConfig.doAssembleCcd=True
isrConfig.doBias=True
isrConfig.doVariance=False
isrConfig.doCrosstalk=False
isrConfig.doBrighterFatter=False
isrConfig.doDark=False
isrConfig.doStrayLight=False
isrConfig.doFlat=False
isrConfig.doFringe=False
isrConfig.doApplyGains=True
isrConfig.usePtcGains=True
isrConfig.doDefect=False
isrConfig.doNanMasking=True
isrConfig.doInterpolate=False
isrConfig.doSaturation=False
isrConfig.doSaturationInterpolation=False
isrTask = IsrTask(config=isrConfig)

## Run the ISR and look at the result

In [ ]:
expId = 2024111600300
exp = butler.get('raw', detector=4, exposure=expId, instrument=instrument)
biasExp = butler.get('bias', detector=4, exposure=expId, instrument=instrument) # This is a bias image associated with the data
ptcExp = butler.get('ptc', detector=4, exposure=expId, instrument=instrument) # This is a bias image associated with the data
isrResult = isrTask.run(exp, bias=biasExp, ptc=ptcExp) # This runs the ISR
x = plot(isrResult.exposure, stretch='ccs')
#plt.savefig(f"/home/c/cslage/u/ComCam/images/ComCam_{expId}.png")

# Plot a small region

In [ ]:
x = 850; y = 3350
width = 200
center = Point2D(x, y)
extent = Extent2I(width, width)
cutout = exp.getCutout(center, extent)
x = plot(cutout, stretch='ccs', showCompass=False)

In [ ]:
plt.title(f"ComCam {expId}, Bias")
plt.plot(isrResult.exposure.image.array[2100, :], label='X cut')
plt.plot(isrResult.exposure.image.array[:, 2100], label='Y cut')
#plt.ylim(0,70000)
plt.ylim(-50, 50)
plt.ylabel("Flux (electrons)")
plt.xlabel("Pixels")
plt.legend()
plt.savefig(f"/home/c/cslage/u/ComCam/images/ComCam_Slices_{expId}.png")

# Now assemble all 9 CCDs and plot the result

In [ ]:
def isrCallback(im, ccd, imageSource):
    """Assemble the CCD image and do basic ISR"""
    dayObs = imageSource.kwargs['day_obs']
    seqNum = imageSource.kwargs['seq_num']
    exp = imageSource.butler.get('raw', detector=ccd.getId(), day_obs=dayObs, seq_num=seqNum)
    biasExp = imageSource.butler.get('bias', detector=ccd.getId(), day_obs=dayObs, seq_num=seqNum)
    ptcExp = butler.get('ptc', detector=ccd.getId(), day_obs=dayObs, 
                        seq_num=seqNum, exposure=expId, instrument=instrument)
    isrResult = isrTask.run(exp, bias=biasExp, ptc=ptcExp) # This runs the ISR
    oim = isrResult.exposure.image
    return oim

def simpleCallback(im, ccd, imageSource):
    """Assemble the CCD image."""
    oim = camGeomUtils.rawCallback(im, ccd, imageSource,
                                   subtractBias=False, correctGain=False)
    return oim

In [ ]:
%matplotlib inline
instrument = "LSSTComCam"
camera = butler.get('camera', instrument=instrument)
fig = plt.figure(figsize=(12,12))
import lsst.afw.display as afwDisplay
disp = afwDisplay.Display(1, "matplotlib")
disp.scale('linear', min='zscale')
dayObs = 20241116
seqNum = 300
dataType='raw'
mos = camGeomUtils.showCamera(camera,
                              camGeomUtils.ButlerImage(butler, dataType, 
                                                       instrument=instrument, raft="R22",
                                                       day_obs=dayObs, seq_num=seqNum,
                                                       verbose=False, callback=isrCallback,
                                                       background=np.nan),
                              binSize=4, display=disp, overlay=False,
                              title="%d %d" % (dayObs, seqNum))

#plt.savefig(f"/home/c/cslage/u/ComCam/images/ComCam_{dayObs}_{seqNum}.png")

# The cell below will save the combined image as a FITS file

In [ ]:
filename = "/home/c/cslage/u/ComCam/images/FITS_2024111600300.fits"
mos.writeFits(filename)